# 01 — Apache Iceberg erkunden: Was steckt in einer Tabelle?

`make seed` hat 5 Tabellen im Raw Layer angelegt. In diesem Notebook schauen wir uns an, was Iceberg dabei eigentlich auf dem Storage abgelegt hat — Parquet-Dateien, Manifest-Indexe, Snapshots, Metadaten.

Das ist keine abstrakte Theorie, sondern eine Expedition in die echten Dateien die gerade auf MinIO liegen.  
Am Ende weiß man warum Iceberg mehr ist als "Parquet in einem Ordner".

In [1]:
import sys
sys.path.insert(0, "/home/jovyan/notebooks")
from spark_init import get_spark_session, trino_query

spark = get_spark_session("01-iceberg-erkunden")

Lakehouse Helpers geladen ✅


---
## Was liegt im Raw Layer?

Nessie ist unser Iceberg-Catalog — er kennt alle Tabellen, ihre Locations auf S3 und ihre Versionshistorie.  
Fragen wir ihn zuerst was er weiß.

In [9]:
spark.sql("SHOW TABLES IN nessie.raw").show()

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|      raw|     cdp_emissions|      false|
|      raw|       fund_master|      false|
|      raw|    fund_positions|      false|
|      raw|   nzdpu_emissions|      false|
|      raw|owid_co2_countries|      false|
+---------+------------------+-----------+



In [2]:
# fund_positions als Beispiel — übersichtliche Größe, hat Partitionierung
spark.sql("SELECT * FROM nessie.raw.fund_positions LIMIT 10").show()

total = spark.sql("SELECT count(*) FROM nessie.raw.fund_positions").collect()[0][0]
print(f"Gesamtzeilen: {total}")

+------------+------------+----------+-------------+
|   fund_isin|holding_isin|weight_pct|position_date|
+------------+------------+----------+-------------+
|DE000DK0EC05|IE00BZ12WP82|    0.9067|   2023-12-31|
|DE000DK0EC05|GB0009895292|   10.6432|   2023-12-31|
|DE000DK0EC05|CH0011075394|    5.2953|   2023-12-31|
|DE000DK0EC05|FR0000120578|     2.293|   2023-12-31|
|DE000DK0EC05|DE0007100000|      8.72|   2023-12-31|
|DE000DK0EC05|NL0010273215|   10.5236|   2023-12-31|
|DE000DK0EC05|DE000BAY0017|    8.1116|   2023-12-31|
|DE000DK0EC05|GB0007980591|    5.3871|   2023-12-31|
|DE000DK0EC05|FR0000121014|   11.3066|   2023-12-31|
|DE000DK0EC05|DE0005190003|    8.9254|   2023-12-31|
+------------+------------+----------+-------------+

Gesamtzeilen: 319


---
## Die Anatomie einer Iceberg-Tabelle

Eine Iceberg-Tabelle ist **keine einzelne Datei** und kein simples Verzeichnis.  
Sie besteht aus vier Schichten — jede mit einer eigenen Aufgabe:

```
s3://raw/fund_positions/
├── data/
│   ├── position_date=2023-12-31/
│   │   └── 00001.parquet      ← Data Files: die eigentlichen Zeilen
│   └── position_date=2024-06-30/
│       └── 00002.parquet
└── metadata/
    ├── 00000-....metadata.json  ← Metadata File: Schema, Partitionen, Snapshot-Liste
    ├── snap-...avro             ← Manifest List: welche Manifests gehören zu diesem Snapshot
    └── ...-m0.avro             ← Manifest File: welche Data Files gibt es, mit Statistiken
```

Schauen wir uns jede Schicht live an.

### Data Files — wo liegen die eigentlichen Daten?

Iceberg führt eine exakte Liste aller Parquet-Dateien die zur Tabelle gehören.  
Das ist kein Verzeichnis-Scan wie bei Hive — Iceberg **weiß** welche Dateien da sind, ohne nachschauen zu müssen.

In [4]:
spark.sql("""
    SELECT file_path,
           file_format,
           record_count,
           file_size_in_bytes,
           round(file_size_in_bytes / 1024.0, 1) AS size_kb
    FROM nessie.raw.fund_positions.files
""").show(truncate=False)

print("\u261d\ufe0f Jede Zeile ist eine Parquet-Datei auf MinIO.")
print("   Iceberg kennt Pfad, Format, Zeilen-Anzahl und Dateigröße — ohne die Datei zu öffnen.")

+--------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+-------+
|file_path                                                                                                           |file_format|record_count|file_size_in_bytes|size_kb|
+--------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+-------+
|s3a://raw/fund_positions/data/position_date=2023-12-31/00000-30-605f3601-9bad-456f-868a-2396a18b10f6-0-00001.parquet|PARQUET    |150         |2963              |2.9    |
|s3a://raw/fund_positions/data/position_date=2024-06-30/00000-30-605f3601-9bad-456f-868a-2396a18b10f6-0-00002.parquet|PARQUET    |169         |3078              |3.0    |
+--------------------------------------------------------------------------------------------------------------------+-----------+------------+--

### Manifest Files — der Statistik-Index über die Data Files

Manifests sind Icebergs Schlüssel zu schnellen Abfragen.  
Für jede Data File speichern sie: Zeilen-Anzahl, Null-Counts, **Min/Max-Werte pro Spalte**.

Fragt jemand `WHERE position_date = '2024-06-30'`, schaut Iceberg ins Manifest und weiß sofort:  
→ Datei 1 enthält nur `2023-12-31`-Daten → überspringen  
→ Datei 2 enthält `2024-06-30`-Daten → lesen  

Das nennt sich **Predicate Pushdown** — ohne die Datei überhaupt zu öffnen.

In [5]:
spark.sql("""
    SELECT path,
           length,
           partition_spec_id,
           added_snapshot_id,
           added_data_files_count,
           existing_data_files_count
    FROM nessie.raw.fund_positions.manifests
""").show(truncate=False)

+------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+
|path                                                                          |length|partition_spec_id|added_snapshot_id  |added_data_files_count|existing_data_files_count|
+------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+
|s3a://raw/fund_positions/metadata/ab588113-fd39-4f97-a9c5-86647b3aa62f-m0.avro|7239  |0                |6817756873344267024|2                     |0                        |
+------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+



### Snapshots — die Versionshistorie der Tabelle

Jede Schreiboperation erzeugt automatisch einen neuen **Snapshot** — einen konsistenten,  
unveränderlichen Zustand der Tabelle zu einem bestimmten Zeitpunkt.

Alte Snapshots bleiben erhalten. Das ist die Basis für **Time Travel**: Man kann jederzeit  
fragen: "Wie sah diese Tabelle vor 3 Monaten aus?" — ohne Backup, ohne Restore.

In [6]:
from pyspark.sql.functions import col

df_snaps = spark.sql("""
    SELECT snapshot_id,
           date_format(committed_at, 'yyyy-MM-dd HH:mm:ss') AS committed_at,
           operation,
           summary['added-records']     AS neue_zeilen,
           summary['total-records']     AS gesamt_zeilen,
           summary['added-data-files']  AS neue_dateien,
           summary['engine-name']       AS engine,
           summary['iceberg-version']   AS iceberg
    FROM nessie.raw.fund_positions.snapshots
    ORDER BY committed_at
""")

df_snaps.show(truncate=False)

print("Jede Operation (append, overwrite, delete) wird als Snapshot festgehalten.")
print("Das passiert automatisch — kein manuelles Versionieren nötig.")

+-------------------+-------------------+---------+-----------+-------------+------------+------+----------------------------------------------------------------------+
|snapshot_id        |committed_at       |operation|neue_zeilen|gesamt_zeilen|neue_dateien|engine|iceberg                                                               |
+-------------------+-------------------+---------+-----------+-------------+------------+------+----------------------------------------------------------------------+
|6817756873344267024|2026-03-28 16:44:50|append   |319        |319          |2           |spark |Apache Iceberg 1.7.1 (commit 4a432839233f2343a9eae8255532f911f06358ef)|
+-------------------+-------------------+---------+-----------+-------------+------------+------+----------------------------------------------------------------------+

Jede Operation (append, overwrite, delete) wird als Snapshot festgehalten.
Das passiert automatisch — kein manuelles Versionieren nötig.


### Schema — Iceberg kennt sein eigenes Schema

Anders als ein reines Parquet-Verzeichnis speichert Iceberg das Schema **in den Metadaten**,  
nicht in den Datendateien. Das ermöglicht Schema Evolution: Spalten umbenennen, Typen  
erweitern — ohne die Parquet-Dateien anzufassen.

In [7]:
spark.sql("DESCRIBE EXTENDED nessie.raw.fund_positions").show(50, truncate=False)

+----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                                                                                                                                                                              |comment|
+----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----

### Partitionierung — versteckt, aber wirksam

`fund_positions` wurde beim Ingest nach `position_date` partitioniert.  
**Iceberg versteckt das vor dem Benutzer** — man schreibt `WHERE position_date = '2024-06-30'`  
ohne zu wissen wie die Tabelle partitioniert ist. Iceberg pruned automatisch die richtigen Dateien.

In [8]:
spark.sql("""
    SELECT partition, record_count, file_count
    FROM nessie.raw.fund_positions.partitions
""").show(truncate=False)

+------------+------------+----------+
|partition   |record_count|file_count|
+------------+------------+----------+
|{2023-12-31}|150         |1         |
|{2024-06-30}|169         |1         |
+------------+------------+----------+



---
## Vergleich: Alle 5 Tabellen im Überblick

Jede Tabelle hat ein anderes Profil — unterschiedliche Größen, Dateien, Partitionierung.  
Schauen wir uns das aggregiert an.

In [9]:
tables = ["nzdpu_emissions", "cdp_emissions", "owid_co2_countries", "fund_master", "fund_positions"]

print(f"{'Tabelle':<25} {'Zeilen':>8} {'Dateien':>8} {'Größe (KB)':>12} {'Snapshots':>10}")
print("\u2500" * 68)

for t in tables:
    rows  = spark.sql(f"SELECT count(*)                    FROM nessie.raw.{t}").collect()[0][0]
    files = spark.sql(f"SELECT count(*)                    FROM nessie.raw.{t}.files").collect()[0][0]
    size  = spark.sql(f"SELECT sum(file_size_in_bytes)     FROM nessie.raw.{t}.files").collect()[0][0] or 0
    snaps = spark.sql(f"SELECT count(*)                    FROM nessie.raw.{t}.snapshots").collect()[0][0]
    print(f"{t:<25} {rows:>8} {files:>8} {round(size / 1024, 1):>12} {snaps:>10}")

print()
print("owid_co2_countries hat 5 Parquet-Dateien → 5 Partitionen (year=2020 bis year=2024).")
print("fund_master hat 1 Datei, keine Partitionierung — klein genug um alles auf einmal zu lesen.")

Tabelle                     Zeilen  Dateien   Größe (KB)  Snapshots
────────────────────────────────────────────────────────────────────


nzdpu_emissions                 90        1          6.2          1


cdp_emissions                  100        1          8.4          1


owid_co2_countries             100        5        172.8          1


fund_master                     10        1          2.5          1


fund_positions                 319        2          5.9          1

owid_co2_countries hat 5 Parquet-Dateien → 5 Partitionen (year=2020 bis year=2024).
fund_master hat 1 Datei, keine Partitionierung — klein genug um alles auf einmal zu lesen.


---
## Wo liegen die Dateien physisch?

Alle Iceberg-Dateien liegen als Objekte in **MinIO** — dem S3-kompatiblen Objektspeicher dieser Umgebung.  
In der Produktion wäre das FI-TS S3 oder ein anderer S3-kompatibler Store.

Du kannst die Dateien direkt in der MinIO Console anschauen:  
→ **http://localhost:9001** (Login: `lakehouse` / `lakehouse123`)  
→ Bucket `raw` → z.B. `fund_positions/` → `data/` und `metadata/`

In [10]:
# Pfade der Data Files — damit man in MinIO nachschauen kann
print("Data Files von fund_positions (vollständige S3-Pfade):")
spark.sql("SELECT file_path FROM nessie.raw.fund_positions.files").show(truncate=False)

print("Data Files von owid_co2_countries (partitioniert nach year):")
spark.sql("SELECT file_path FROM nessie.raw.owid_co2_countries.files").show(truncate=False)

print("Diese Pfade kannst du 1:1 in der MinIO Console unter Bucket 'raw' nachschauen.")

Data Files von fund_positions (vollständige S3-Pfade):
+--------------------------------------------------------------------------------------------------------------------+
|file_path                                                                                                           |
+--------------------------------------------------------------------------------------------------------------------+
|s3a://raw/fund_positions/data/position_date=2023-12-31/00000-30-605f3601-9bad-456f-868a-2396a18b10f6-0-00001.parquet|
|s3a://raw/fund_positions/data/position_date=2024-06-30/00000-30-605f3601-9bad-456f-868a-2396a18b10f6-0-00002.parquet|
+--------------------------------------------------------------------------------------------------------------------+

Data Files von owid_co2_countries (partitioniert nach year):


+---------------------------------------------------------------------------------------------------------+
|file_path                                                                                                |
+---------------------------------------------------------------------------------------------------------+
|s3a://raw/owid_co2_countries/data/year=2024/00000-15-1100c65b-0ad2-488e-8de9-a167f0c16648-0-00004.parquet|
|s3a://raw/owid_co2_countries/data/year=2022/00000-15-1100c65b-0ad2-488e-8de9-a167f0c16648-0-00002.parquet|
|s3a://raw/owid_co2_countries/data/year=2023/00000-15-1100c65b-0ad2-488e-8de9-a167f0c16648-0-00001.parquet|
|s3a://raw/owid_co2_countries/data/year=2020/00000-15-1100c65b-0ad2-488e-8de9-a167f0c16648-0-00003.parquet|
|s3a://raw/owid_co2_countries/data/year=2021/00000-15-1100c65b-0ad2-488e-8de9-a167f0c16648-0-00005.parquet|
+---------------------------------------------------------------------------------------------------------+

Diese Pfade kannst du 1:1 i

---
## Zusammenfassung

Was wir gesehen haben:

| Schicht | Datei-Typ | Aufgabe |
|---|---|---|
| Data Files | `.parquet` | Die eigentlichen Datenzeilen |
| Manifest Files | `.avro` | Index mit Statistiken (Min/Max, Nulls) je Data File |
| Manifest List | `.avro` | Verknüpft Manifests mit einem Snapshot |
| Metadata File | `.json` | Schema, Partitionierung, Snapshot-Historie |

**Warum das clever ist:**
- Iceberg muss keine Verzeichnisse scannen — es liest die Manifest-Statistiken und überspringt irrelevante Dateien
- Snapshots entstehen automatisch bei jeder Schreiboperation — Time Travel ist keine Zusatzfunktion, sondern eingebaut
- Das Schema lebt in den Metadaten, nicht in den Parquet-Dateien — Schema Evolution ohne Datenmigration
- Alle Dateien liegen auf S3 — jede Engine (Spark, Trino, DuckDB) kann sie lesen

---
→ **Notebook 02:** Time Travel und Schema Evolution — was man mit diesen Snapshots konkret machen kann